# CIFAR-10 Autoencoder

Training a convolutional autoencoder to reconstruct CIFAR-10 images with PyTorch Lightning.

In [ ]:
%matplotlib inline
%config InlineBackend.figure_formats = ['retina', 'svg']
%load_ext autoreload
%autoreload 2

import os

import matplotlib.pyplot as plt
import torch
from lightning import Trainer, seed_everything
from torchinfo import summary

from chimera.data import CIFAR10DataModule
from chimera.modules import AutoencoderModule
from chimera.optim import LinearWarmupCosineAnnealingLR

os.environ["DATA_DIR"] = "../../../data"

# Reproducibility: seed all RNGs (incl. dataloader workers). Pair with
# Trainer(deterministic=True) below for deterministic CUDA kernels too.
SEED = 42
seed_everything(SEED, workers=True)

## Data

Load CIFAR-10. Pixels stay in `[0, 1]`, matching the decoder's sigmoid output.

In [ ]:
dm = CIFAR10DataModule(
    pin_memory=False,
    num_workers=0,
    data_dir=os.environ["DATA_DIR"],
)
dm.prepare_data()
dm.setup("fit")

train_loader = dm.train_dataloader()
val_loader = dm.val_dataloader()

In [ ]:
images, _ = next(iter(train_loader))

fig, axes = plt.subplots(1, 8, figsize=(12, 2))
fig.suptitle("Sample CIFAR-10 Images")
for ax, img in zip(axes, images):
    ax.imshow(img.permute(1, 2, 0))
    ax.axis("off")
plt.show()

## Model

A convolutional autoencoder for CIFAR: the encoder compresses each 3x32x32 image to a latent vector, and the decoder reconstructs it.

In [ ]:
from chimera.models import CIFARAutoencoder

model = CIFARAutoencoder(in_channels=3, latent_dim=128)
summary(
    model,
    input_size=(1, 3, 32, 32),
    col_names=["output_size", "mult_adds", "num_params"],
    verbose=0,
)

## Training

Wrap the model in a `LightningModule` and train with a linear-warmup cosine-annealing schedule, minimizing reconstruction MSE.

In [ ]:
N_EPOCH = 20

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
scheduler = LinearWarmupCosineAnnealingLR(
    optimizer,
    warmup_steps=1000,
    n_epochs=N_EPOCH,
    train_loader=train_loader,
)

autoencoder_module = AutoencoderModule(model, optimizer, scheduler)

trainer = Trainer(
    max_epochs=N_EPOCH,
    precision="bf16-true",
    deterministic=True,
)

trainer.fit(
    autoencoder_module, train_dataloaders=train_loader, val_dataloaders=val_loader
)
trainer.test(autoencoder_module, dataloaders=val_loader)

## Analysis

Plot the logged reconstruction loss curves, then compare original images against their reconstructions.

In [ ]:
import numpy as np

metrics = np.genfromtxt(
    f"{trainer.logger.log_dir}/metrics.csv", delimiter=",", names=True
)


def series(step_key, value_key):
    step = metrics[step_key]
    value = metrics[value_key]
    mask = ~np.isnan(value)
    return step[mask], value[mask]


plt.figure(figsize=(7, 5))
plt.title("Reconstruction Loss")
train_step, train_val = series("step", "trainloss")
val_step, val_val = series("step", "valloss")
plt.plot(train_step, train_val, marker="o", label="train")
plt.plot(val_step, val_val, marker="o", label="val")
plt.xlabel("Step")
plt.ylabel("MSE")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
autoencoder_module.eval()
device = autoencoder_module.device

images, _ = next(iter(val_loader))
with torch.no_grad():
    recons = (
        autoencoder_module(images.to(device, dtype=autoencoder_module.dtype))
        .float()
        .cpu()
    )

n = 8
fig, axes = plt.subplots(2, n, figsize=(12, 3))
fig.suptitle("Original (top) vs Reconstruction (bottom)")
for i in range(n):
    axes[0, i].imshow(images[i].permute(1, 2, 0).clamp(0, 1))
    axes[0, i].axis("off")
    axes[1, i].imshow(recons[i].permute(1, 2, 0).clamp(0, 1))
    axes[1, i].axis("off")
plt.show()